## Pilot Run – Movie Reviews (archived)

This notebook was a pilot run used to validate the BERTurk training pipeline
before running all domains.

**Validation results (best epoch = 3):** Accuracy 0.9446, Macro-F1 0.9446

**Why it was archived:**
- Checkpoints were saved to local Colab storage and lost when the runtime terminated.
- The test set was not evaluated.
- Precision/recall used binary averaging instead of macro averaging.

**Changes applied in the final notebook:** macro-averaged metrics, checkpoints and
models saved to Drive with automatic resume, fresh model per domain, evaluation on
all four test sets (in-domain + cross-domain), saved predictions for error analysis,
fp16 and dynamic padding.

In [3]:
!pip install -q transformers datasets evaluate accelerate

In [4]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [5]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
train_df = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/train_4domain_natural.csv"
)

val_df = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/val_4domain_natural.csv"
)

test_df = pd.read_csv(
    "/content/drive/MyDrive/crossDomainSentAnlys/test_4domain_natural.csv"
)

In [10]:
MODEL_NAME = "dbmdz/bert-base-turkish-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

model.to(device)

cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [12]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [13]:
!nvidia-smi

Mon Nov 16 12:08:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             26W /   70W |     581MiB /  15360MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [15]:
!nvidia-smi

Mon Nov 16 12:08:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             26W /   70W |     581MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
def get_domain_data(domain_name):

    train_domain = train_df[
        train_df["Domain"] == domain_name
    ].copy()

    val_domain = val_df[
        val_df["Domain"] == domain_name
    ].copy()

    test_domain = test_df[
        test_df["Domain"] == domain_name
    ].copy()

    return train_domain, val_domain, test_domain

In [17]:
train_movie, val_movie, test_movie = get_domain_data(
    "Movie Reviews"
)

print(train_movie.shape)
print(val_movie.shape)
print(test_movie.shape)

(31360, 4)
(4480, 4)
(8960, 4)


In [18]:
train_movie.columns

Index(['text', 'label', 'dataset', 'Domain'], dtype='object')

In [21]:
train_movie.describe(include='object')

,text,label,dataset,Domain
count,31360,31360,31360,31360
unique,31360,2,1,1
top,Daha iyi olabilirmiş. Türk sinemasının çok ön...,Negative,HUMIR,Movie Reviews
freq,1,15695,31360,31360


Since BERT does not get along very well with string labels, we are transforming "negative" label to 0 and "positive" label to 1 by adding another column

In [22]:
label_map = {
    "Negative": 0,
    "Positive": 1
}

def encode_labels(df):

    df = df.copy()

    df["label_id"] = df["label"].map(
        label_map
    )

    return df

In [23]:
train_movie = encode_labels(
    train_movie
)

train_movie.head()

,text,label,dataset,Domain,label_id
0,Daha iyi olabilirmiş. Türk sinemasının çok ön...,Negative,HUMIR,Movie Reviews,0
3,Hani tilki acmeden paket alırdı road runner ı...,Negative,HUMIR,Movie Reviews,0
8,Saçma bulduum filmlerin arasına koyabileceim ...,Negative,HUMIR,Movie Reviews,0
14,süperr harika mühteşem bi film kesinlikle osc...,Positive,HUMIR,Movie Reviews,1
20,Kesinlikle çok kötü bir film.Benim en kötü ve...,Negative,HUMIR,Movie Reviews,0


In [24]:
from datasets import Dataset

def to_hf_dataset(df):

    return Dataset.from_pandas(
        df[["text", "label_id"]]
    )

In [25]:
train_dataset = to_hf_dataset(
    train_movie
)

train_dataset


Dataset({
    features: ['text', 'label_id', '__index_level_0__'],
    num_rows: 31360
})

In [27]:
MAX_LENGTH = 256

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [28]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_train

Map:   0%|          | 0/31360 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label_id', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 31360
})

In [29]:
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val

NameError: name 'val_dataset' is not defined

In [30]:
val_movie = encode_labels(
    val_movie
)

test_movie = encode_labels(
    test_movie
)

In [31]:
val_dataset = to_hf_dataset(
    val_movie
)

test_dataset = to_hf_dataset(
    test_movie
)

In [33]:
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/4480 [00:00<?, ? examples/s]

Map:   0%|          | 0/8960 [00:00<?, ? examples/s]

In [34]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")


In [35]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    precision = precision_metric.compute(
        predictions=predictions,
        references=labels
    )

    recall = recall_metric.compute(
        predictions=predictions,
        references=labels
    )

    macro_f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "macro_f1": macro_f1["f1"]
    }

In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./berturk_movie",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    seed=42,

    logging_steps=100
)

In [37]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    compute_metrics=compute_metrics
)

In [39]:
tokenized_train = tokenized_train.rename_column(
    "label_id",
    "labels"
)

tokenized_val = tokenized_val.rename_column(
    "label_id",
    "labels"
)

tokenized_test = tokenized_test.rename_column(
    "label_id",
    "labels"
)

In [40]:
tokenized_train

Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 31360
})

In [41]:
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

In [42]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)


In [43]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1,0.161706,0.247527,0.933929,0.906616,0.967382,0.933858
2,0.125892,0.238827,0.941964,0.924099,0.962913,0.941941
3,0.070901,0.259451,0.944643,0.937170,0.953083,0.944640


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5880, training_loss=0.13505568544880872, metrics={'train_runtime': 4049.9243, 'train_samples_per_second': 23.23, 'train_steps_per_second': 1.452, 'total_flos': 1.23767440441344e+16, 'train_loss': 0.13505568544880872, 'epoch': 3.0})

In [1]:
import os

os.listdir()

['.config', 'sample_data']

In [2]:
!ls


sample_data


In [3]:
!find . -type d -name "checkpoint*"

In [ ]:
!ls /content

In [1]:
!ls /content

sample_data


In [8]:
print(train_df["Domain"].unique())
print(train_df.groupby(["Domain", "label"]).size())

['Movie Reviews' 'Product Reviews' 'Hotel Reviews' 'Tweets']
Domain           label   
Hotel Reviews    Negative      4345
                 Positive      3322
Movie Reviews    Negative     15695
                 Positive     15665
Product Reviews  Negative     10347
                 Positive    153476
Tweets           Negative      3476
                 Positive      4259
dtype: int64


In [ ]:
def get_domain_data(domain_name):

    train_domain = train_df[
        train_df["Domain"] == domain_name
    ].copy()

    val_domain = val_df[
        val_df["Domain"] == domain_name
    ].copy()

    test_domain = test_df[
        test_df["Domain"] == domain_name
    ].copy()

    return train_domain, val_domain, test_domain, .head(64)